In [7]:
%%writefile csunet_catdog_s3.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CSUNet-SSR-S3 — ESP32-S3 / ESP32-S3-CAM oriented Cat-vs-Dog model
===================================================================

Purpose
-------
Hardware-aware student derived from CSUNet-SSR. The model keeps the distinctive
shared/specific convolutional sparse-unrolling residual classifier, but compresses
encoder width, spatial resolution, dictionary size and unrolling depth for ESP32-S3.

Main design (default balanced profile)
--------------------------------------
Input 128x128
  -> Conv3x3 s2 3->16
  -> multi-scale DW(3x3)+DW(5x5) additive stem, PW 16->24
  -> DSConv s2 24->32, DSConv 32->32
  -> DSConv s2 32->48, DSConv 48->48
  -> DSConv s2 48->64, DSConv 64->64
  -> PW 64->12 => feature map 12x8x8
  -> shared/specific LISTA-style unrolling:
       D = [D_shared | D_cat | D_dog]
       shared atoms=2, atoms/class=4, kernel=3x3, stages=4
  -> class residual after shared-content cancellation

ESP-friendly details
--------------------
* ReLU only in the student encoder (no SiLU/SE in the student).
* Soft-threshold is written exactly as:
      relu(z - tau) - relu(-z - tau)
  so Sign/Abs are not needed in the deploy graph.
* Deploy export freezes normalized D/W, learned steps, thresholds and logit scale.
* ConvTranspose2d synthesis is algebraically converted to ordinary Conv2d with a
  flipped/transposed kernel for the frozen ONNX deploy graph.
* Static batch=1 ONNX opset 18 export.

Overfitting controls
--------------------
* Fixed stratified train/val/test split (default 80/10/10, seed 42).
* Test is locked unless --unlock_test is explicitly used.
* Augmentation, AdamW, weight decay, label smoothing, EMA.
* Best checkpoint selected by validation macro-F1.
* Early stopping uses --min_delta, not microscopic numerical improvements.
* Optional knowledge distillation from the full CSUNet-SSR teacher.
* Train-val gap is logged every epoch; optional RandomErasing remains OFF by default.

Recommended Kaggle command after the full teacher is trained
-------------------------------------------------------------
python /kaggle/working/csunet_catdog_s3.py \
  --data_root /kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages \
  --teacher_ckpt /kaggle/working/csunet_ssr/csunet_ssr_best.pt \
  --out_dir /kaggle/working/csunet_ssr_s3 \
  --img_size 128 --teacher_img_size 160 \
  --widths 16,24,32,48,64 --feat_ch 12 \
  --shared_atoms 2 --apc 4 --ksize 3 --layers 4 \
  --batch_size 128 --epochs 70 --lr 1.5e-3 --weight_decay 5e-4 \
  --warmup_epochs 4 --label_smoothing 0.05 \
  --kd_weight 0.40 --kd_temp 3.0 --kd_ramp_epochs 5 \
  --ema_decay 0.999 --patience 8 --min_delta 0.001 \
  --w_recon 0.05 --w_cross 0.05 --w_margin 0.20 --margin 0.10 --w_sep 0.01 \
  --amp

After architecture/hyperparameters are FROZEN, unlock test once:
python .../csunet_catdog_s3.py <same model args> \
  --out_dir /kaggle/working/csunet_ssr_s3 --eval_only --unlock_test --export_onnx

Scientific note
---------------
The learned-step unrolling is LISTA-style. Do not claim the classical ISTA
convergence guarantee: W may be untied and steps are learned/bounded.
"""

import argparse
import copy
import csv
import json
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset

try:
    from torchvision import transforms
    from torchvision.transforms import functional as TF
except Exception as e:
    raise RuntimeError("torchvision is required") from e

try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

ImageFile.LOAD_TRUNCATED_IMAGES = True

CLASS_NAMES = ["Cat", "Dog"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
STUDENT_MEAN = (0.5, 0.5, 0.5)
STUDENT_STD = (0.5, 0.5, 0.5)
TEACHER_MEAN = (0.485, 0.456, 0.406)
TEACHER_STD = (0.229, 0.224, 0.225)


# ============================================================================
# Reproducibility
# ============================================================================
def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


# ============================================================================
# Dataset discovery / split
# ============================================================================
def _label_from_path(path: Path) -> Optional[int]:
    parts = [p.lower() for p in path.parts]
    for p in reversed(parts[:-1]):
        if p in {"cat", "cats"}:
            return 0
        if p in {"dog", "dogs"}:
            return 1
    name = path.name.lower()
    if name.startswith(("cat.", "cat_", "cat-")):
        return 0
    if name.startswith(("dog.", "dog_", "dog-")):
        return 1
    return None


def _scan_labeled_images(root: Path) -> List[Tuple[str, int]]:
    out: List[Tuple[str, int]] = []
    if not root.exists():
        return out
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            y = _label_from_path(p)
            if y is not None:
                out.append((str(p), y))
    return sorted(set(out), key=lambda z: z[0])


def _find_train_zip(root: Path) -> Optional[Path]:
    for p in (root / "train.zip", root / "dogs-vs-cats" / "train.zip"):
        if p.exists():
            return p
    zips = sorted(root.rglob("train.zip")) if root.exists() else []
    return zips[0] if zips else None


def discover_catdog_samples(data_root: str, out_dir: str) -> List[Tuple[str, int]]:
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"data_root does not exist: {data_root}")
    samples = _scan_labeled_images(root)
    if samples:
        return samples
    train_zip = _find_train_zip(root)
    if train_zip is None:
        raise FileNotFoundError(
            f"No labeled Cat/Dog images or train.zip found under {data_root}"
        )
    extract_root = Path(out_dir) / "_catdog_extracted"
    marker = extract_root / ".done"
    if not marker.exists():
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f"[DATA] extracting {train_zip} -> {extract_root}", flush=True)
        with zipfile.ZipFile(train_zip, "r") as zf:
            zf.extractall(extract_root)
        marker.write_text("ok\n", encoding="utf-8")
    samples = _scan_labeled_images(extract_root)
    if not samples:
        raise RuntimeError("Extraction succeeded but no labeled Cat/Dog images were found")
    return samples


def maybe_subset(samples: List[Tuple[str, int]], frac: float, seed: int) -> List[Tuple[str, int]]:
    if frac >= 1.0:
        return samples
    if frac <= 0:
        raise ValueError("subset_frac must be > 0")
    rng = np.random.default_rng(seed)
    out: List[Tuple[str, int]] = []
    for c in range(2):
        cls = [s for s in samples if s[1] == c]
        idx = rng.permutation(len(cls))[:max(1, int(round(len(cls) * frac)))]
        out.extend(cls[int(i)] for i in idx)
    rng.shuffle(out)
    return out


def stratified_split(
    samples: Sequence[Tuple[str, int]], seed: int = 42,
    val_frac: float = 0.10, test_frac: float = 0.10,
) -> Tuple[List[Tuple[str, int]], List[Tuple[str, int]], List[Tuple[str, int]]]:
    if val_frac < 0 or test_frac < 0 or val_frac + test_frac >= 1:
        raise ValueError("Require val_frac>=0, test_frac>=0, val_frac+test_frac<1")
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for c in range(2):
        cls = [x for x in samples if x[1] == c]
        if not cls:
            raise RuntimeError(f"No samples for {CLASS_NAMES[c]}")
        perm = rng.permutation(len(cls))
        n_te = int(round(len(cls) * test_frac))
        n_va = int(round(len(cls) * val_frac))
        for rank, j in enumerate(perm):
            item = cls[int(j)]
            if rank < n_te:
                te.append(item)
            elif rank < n_te + n_va:
                va.append(item)
            else:
                tr.append(item)
    rng.shuffle(tr); rng.shuffle(va); rng.shuffle(te)
    return tr, va, te


def print_split_stats(name: str, samples: Sequence[Tuple[str, int]]) -> None:
    cnt = [sum(int(y == c) for _, y in samples) for c in range(2)]
    print(f"[DATA] {name:<5s}: {len(samples):6d} | Cat={cnt[0]:6d} Dog={cnt[1]:6d}", flush=True)


# ============================================================================
# Paired teacher/student transforms
# ============================================================================
class PairedTrainTransform:
    """One stochastic crop/color transform, then two normalizations.

    The teacher sees a teacher_img_size crop (normally 160). The student receives the
    exact same crop downsampled to img_size (normally 128), preventing KD from comparing
    two different random views.
    """
    def __init__(self, img_size: int, teacher_img_size: int = 160, erase_p: float = 0.0):
        self.img_size = int(img_size)
        self.teacher_img_size = int(max(teacher_img_size, img_size))
        self.erase_p = float(erase_p)
        self.geo_photo = transforms.Compose([
            transforms.RandomResizedCrop(
                self.teacher_img_size, scale=(0.70, 1.0), ratio=(0.80, 1.25)
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
            transforms.RandomRotation(degrees=8),
        ])
        self.erase = transforms.RandomErasing(
            p=self.erase_p, scale=(0.02, 0.06), ratio=(0.5, 2.0), value=0.5
        ) if self.erase_p > 0 else None

    def __call__(self, im: Image.Image) -> Tuple[torch.Tensor, torch.Tensor]:
        im = self.geo_photo(im)
        base = TF.to_tensor(im)  # [0,1], teacher-size
        if self.erase is not None:
            base = self.erase(base)
        teacher = TF.normalize(base.clone(), TEACHER_MEAN, TEACHER_STD)
        student_base = F.interpolate(
            base.unsqueeze(0), size=(self.img_size, self.img_size),
            mode="bilinear", align_corners=False
        ).squeeze(0)
        student = TF.normalize(student_base, STUDENT_MEAN, STUDENT_STD)
        return student, teacher


class StudentEvalTransform:
    def __init__(self, img_size: int):
        resize_to = int(round(img_size * 1.10))
        self.tf = transforms.Compose([
            transforms.Resize(resize_to),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(STUDENT_MEAN, STUDENT_STD),
        ])

    def __call__(self, im: Image.Image) -> torch.Tensor:
        return self.tf(im)


class TrainPairDataset(Dataset):
    def __init__(self, samples: Sequence[Tuple[str, int]], transform: PairedTrainTransform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, y = self.samples[idx]
        with Image.open(path) as im:
            xs, xt = self.transform(im.convert("RGB"))
        return xs, xt, int(y)


class EvalDataset(Dataset):
    def __init__(self, samples: Sequence[Tuple[str, int]], transform: StudentEvalTransform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, y = self.samples[idx]
        with Image.open(path) as im:
            x = self.transform(im.convert("RGB"))
        return x, int(y)


# ============================================================================
# Student: ESP32-S3 encoder
# ============================================================================
class ConvBNReLU(nn.Sequential):
    def __init__(self, cin: int, cout: int, k: int = 3, stride: int = 1,
                 groups: int = 1, act: bool = True):
        p = k // 2
        layers: List[nn.Module] = [
            nn.Conv2d(cin, cout, k, stride=stride, padding=p, groups=groups, bias=False),
            nn.BatchNorm2d(cout),
        ]
        if act:
            layers.append(nn.ReLU(inplace=True))
        super().__init__(*layers)


class S3MultiScaleAddStem(nn.Module):
    """DW3 + DW5 -> add -> pointwise. Avoids concat activation expansion."""
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.b3 = ConvBNReLU(cin, cin, 3, groups=cin)
        self.b5 = ConvBNReLU(cin, cin, 5, groups=cin)
        self.fuse = ConvBNReLU(cin, cout, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fuse(self.b3(x) + self.b5(x))


class S3DSConvBlock(nn.Module):
    def __init__(self, cin: int, cout: int, stride: int = 1):
        super().__init__()
        self.dw = ConvBNReLU(cin, cin, 3, stride=stride, groups=cin)
        self.pw = ConvBNReLU(cin, cout, 1)
        self.use_res = stride == 1 and cin == cout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.pw(self.dw(x))
        return x + y if self.use_res else y


class S3FeatureEncoder(nn.Module):
    def __init__(self, feat_ch: int = 12,
                 widths: Tuple[int, int, int, int, int] = (16, 24, 32, 48, 64)):
        super().__init__()
        c0, c1, c2, c3, c4 = widths
        self.pre = ConvBNReLU(3, c0, 3, stride=2)       # 128 -> 64
        self.stem = S3MultiScaleAddStem(c0, c1)         # 64
        self.stage1 = nn.Sequential(
            S3DSConvBlock(c1, c2, stride=2),            # 64 -> 32
            S3DSConvBlock(c2, c2, stride=1),
        )
        self.stage2 = nn.Sequential(
            S3DSConvBlock(c2, c3, stride=2),            # 32 -> 16
            S3DSConvBlock(c3, c3, stride=1),
        )
        self.stage3 = nn.Sequential(
            S3DSConvBlock(c3, c4, stride=2),            # 16 -> 8
            S3DSConvBlock(c4, c4, stride=1),
        )
        self.project = ConvBNReLU(c4, feat_ch, 1)
        self.out_ch = int(feat_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pre(x)
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.project(x)


# ============================================================================
# Shared/specific sparse unrolling
# ============================================================================
def _orthogonal_atom_init(M: int, dim: int, seed: int = 12345) -> torch.Tensor:
    if M > dim:
        raise ValueError(f"Cannot initialize {M} orthogonal atoms in dim={dim}")
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    A = torch.randn(dim, M, generator=g)
    Q, _ = torch.linalg.qr(A, mode="reduced")
    return Q.t().contiguous()


def soft_threshold_esp(z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Exactly equivalent to sign(z)*relu(abs(z)-t), using ESP-friendly ops."""
    return F.relu(z - t) - F.relu(-z - t)


class SharedSpecificS3Head(nn.Module):
    def __init__(self, in_ch: int, num_classes: int = 2, shared_atoms: int = 2,
                 atoms_per_class: int = 4, ksize: int = 3, n_layers: int = 4,
                 step_max: float = 0.15, untied_w: bool = True, init_seed: int = 12345):
        super().__init__()
        if ksize % 2 == 0:
            raise ValueError("ksize must be odd")
        self.in_ch = int(in_ch)
        self.num_classes = int(num_classes)
        self.shared_atoms = int(shared_atoms)
        self.apc = int(atoms_per_class)
        self.ksize = int(ksize)
        self.n_layers = int(n_layers)
        self.step_max = float(step_max)
        self.untied_w = bool(untied_w)
        self.pad = ksize // 2
        self.M = self.shared_atoms + self.num_classes * self.apc
        flat = _orthogonal_atom_init(self.M, in_ch * ksize * ksize, init_seed)
        D0 = flat.view(self.M, in_ch, ksize, ksize)
        self.D = nn.Parameter(D0)
        if self.untied_w:
            self.W = nn.Parameter(D0.clone())
        self.gamma_raw = nn.Parameter(torch.zeros(n_layers))
        self.theta_raw = nn.Parameter(torch.full((n_layers, self.M), -3.0))
        self.logit_scale_raw = nn.Parameter(torch.tensor(0.5413))
        self.bias = nn.Parameter(torch.zeros(num_classes))

    def _unit_D(self) -> torch.Tensor:
        n = self.D.flatten(1).norm(dim=1).view(-1, 1, 1, 1).clamp_min(1e-8)
        return self.D / n

    def _unit_W(self, Dn: torch.Tensor) -> torch.Tensor:
        if not self.untied_w:
            return Dn
        n = self.W.flatten(1).norm(dim=1).view(-1, 1, 1, 1).clamp_min(1e-8)
        return self.W / n

    def _specific_slice(self, c: int) -> slice:
        lo = self.shared_atoms + c * self.apc
        return slice(lo, lo + self.apc)

    def encode(self, fmap: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        Dn = self._unit_D()
        Wn = self._unit_W(Dn)
        B, _, H, W = fmap.shape
        Z = fmap.new_zeros(B, self.M, H, W)
        steps = self.step_max * torch.sigmoid(self.gamma_raw)
        thrs = F.softplus(self.theta_raw)
        for k in range(self.n_layers):
            recon = F.conv_transpose2d(Z, Dn, padding=self.pad)
            grad = F.conv2d(fmap - recon, Wn, padding=self.pad)
            Z = Z + steps[k] * grad
            t = thrs[k].view(1, self.M, 1, 1)
            Z = soft_threshold_esp(Z, t)
        return Z, Dn

    def residuals(self, fmap: torch.Tensor, Z: torch.Tensor, Dn: torch.Tensor) -> torch.Tensor:
        B = fmap.size(0)
        if self.shared_atoms > 0:
            recon_s = F.conv_transpose2d(
                Z[:, :self.shared_atoms], Dn[:self.shared_atoms], padding=self.pad
            )
        else:
            recon_s = torch.zeros_like(fmap)
        res = fmap.new_empty(B, self.num_classes)
        for c in range(self.num_classes):
            idx = self._specific_slice(c)
            recon_c = F.conv_transpose2d(Z[:, idx], Dn[idx], padding=self.pad)
            res[:, c] = (fmap - recon_s - recon_c).square().flatten(1).mean(1)
        return res

    def cross_specific_loss(self, Z: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        Zs = Z[:, self.shared_atoms:]
        B = Zs.size(0)
        e = Zs.square().mean(dim=(2, 3)).view(B, self.num_classes, self.apc).sum(-1)
        total = e.sum(1).clamp_min(1e-8)
        true_e = e.gather(1, labels.view(-1, 1)).squeeze(1)
        return ((total - true_e).clamp_min(0.0) / total).mean()

    def separation_loss(self, Dn: torch.Tensor) -> torch.Tensor:
        V = Dn.flatten(1)
        terms: List[torch.Tensor] = []
        if self.shared_atoms > 0:
            S = V[:self.shared_atoms]
            for c in range(self.num_classes):
                C = V[self._specific_slice(c)]
                terms.append((S @ C.t()).square().mean())
        for a in range(self.num_classes):
            A = V[self._specific_slice(a)]
            for b in range(a + 1, self.num_classes):
                B = V[self._specific_slice(b)]
                terms.append((A @ B.t()).square().mean())
        return torch.stack(terms).mean() if terms else V.new_zeros(())

    def forward(self, fmap: torch.Tensor, labels: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        Z, Dn = self.encode(fmap)
        res = self.residuals(fmap, Z, Dn)
        scale = F.softplus(self.logit_scale_raw)
        out: Dict[str, torch.Tensor] = {
            "logits": -scale * res + self.bias,
            "residuals": res,
        }
        if labels is not None:
            true_res = res.gather(1, labels.view(-1, 1)).squeeze(1)
            rn = res / res.mean(1, keepdim=True).detach().clamp_min(1e-8)
            true_rn = rn.gather(1, labels.view(-1, 1)).squeeze(1)
            mask = F.one_hot(labels, num_classes=self.num_classes).bool()
            wrong_rn = rn.masked_fill(mask, float("inf")).min(1).values
            out["recon"] = true_res.mean()
            out["margin_raw"] = true_rn - wrong_rn
            out["cross"] = self.cross_specific_loss(Z, labels)
            out["sep"] = self.separation_loss(Dn)
        return out


class CSUNetSSRS3(nn.Module):
    def __init__(self, num_classes: int = 2, feat_ch: int = 12,
                 widths: Tuple[int, int, int, int, int] = (16,24,32,48,64),
                 shared_atoms: int = 2, atoms_per_class: int = 4,
                 ksize: int = 3, n_layers: int = 4, step_max: float = 0.15,
                 untied_w: bool = True):
        super().__init__()
        self.num_classes = int(num_classes)
        self.encoder = S3FeatureEncoder(feat_ch=feat_ch, widths=widths)
        self.head = SharedSpecificS3Head(
            self.encoder.out_ch, num_classes, shared_atoms, atoms_per_class,
            ksize, n_layers, step_max, untied_w
        )

    def forward(self, x: torch.Tensor, labels: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        return self.head(self.encoder(x), labels=labels)

    @torch.inference_mode()
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward(x)["logits"].argmax(1)


# ============================================================================
# Teacher architecture: exact B0 model layout, only for KD checkpoint loading
# ============================================================================
class TeacherConvBNAct(nn.Sequential):
    def __init__(self, cin: int, cout: int, k: int = 3, stride: int = 1,
                 groups: int = 1, act: bool = True):
        p = k // 2
        layers: List[nn.Module] = [
            nn.Conv2d(cin, cout, k, stride=stride, padding=p, groups=groups, bias=False),
            nn.BatchNorm2d(cout),
        ]
        if act:
            layers.append(nn.SiLU(inplace=True))
        super().__init__(*layers)


class TeacherLiteSE(nn.Module):
    def __init__(self, ch: int, r: int = 8):
        super().__init__()
        hidden = max(ch // r, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(ch, hidden, 1)
        self.fc2 = nn.Conv2d(hidden, ch, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.pool(x)
        s = F.silu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s


class TeacherMultiScaleSpatialStem(nn.Module):
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.b3 = TeacherConvBNAct(cin, cin, 3, groups=cin)
        self.b5 = TeacherConvBNAct(cin, cin, 5, groups=cin)
        self.fuse = TeacherConvBNAct(cin * 2, cout, 1)
        self.se = TeacherLiteSE(cout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.se(self.fuse(torch.cat([self.b3(x), self.b5(x)], dim=1)))


class TeacherDSConvBlock(nn.Module):
    def __init__(self, cin: int, cout: int, stride: int = 1):
        super().__init__()
        self.dw = TeacherConvBNAct(cin, cin, 3, stride=stride, groups=cin)
        self.pw = TeacherConvBNAct(cin, cout, 1)
        self.use_res = stride == 1 and cin == cout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.pw(self.dw(x))
        return x + y if self.use_res else y


class TeacherNaturalFeatureEncoder(nn.Module):
    def __init__(self, feat_ch: int = 16, target_hw: int = 12,
                 widths: Tuple[int,int,int,int,int] = (32,64,96,128,160)):
        super().__init__()
        c0,c1,c2,c3,c4 = widths
        self.pre = TeacherConvBNAct(3, c0, 3, stride=2)
        self.stem = TeacherMultiScaleSpatialStem(c0, c1)
        self.stage1 = nn.Sequential(
            TeacherDSConvBlock(c1,c2,stride=2), TeacherDSConvBlock(c2,c2,1)
        )
        self.stage2 = nn.Sequential(
            TeacherDSConvBlock(c2,c3,stride=2), TeacherDSConvBlock(c3,c3,1)
        )
        self.stage3 = nn.Sequential(
            TeacherDSConvBlock(c3,c4,1), TeacherDSConvBlock(c4,c4,1)
        )
        self.project = TeacherConvBNAct(c4, feat_ch, 1)
        self.pool = nn.AdaptiveAvgPool2d((target_hw,target_hw))
        self.post_bn = nn.BatchNorm2d(feat_ch)
        self.out_ch = feat_ch
        self.target_hw = target_hw

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x=self.pre(x); x=self.stem(x); x=self.stage1(x); x=self.stage2(x); x=self.stage3(x)
        x=self.project(x); x=self.pool(x)
        return self.post_bn(x)


class TeacherSharedSpecificHead(nn.Module):
    def __init__(self, in_ch: int, num_classes: int = 2, shared_atoms: int = 4,
                 atoms_per_class: int = 6, ksize: int = 5, n_layers: int = 5,
                 step_max: float = 0.15, untied_w: bool = True, init_seed: int = 12345):
        super().__init__()
        self.in_ch=in_ch; self.num_classes=num_classes; self.shared_atoms=shared_atoms
        self.apc=atoms_per_class; self.ksize=ksize; self.n_layers=n_layers
        self.step_max=step_max; self.untied_w=untied_w; self.pad=ksize//2
        self.M=shared_atoms+num_classes*atoms_per_class
        flat=_orthogonal_atom_init(self.M,in_ch*ksize*ksize,init_seed)
        D0=flat.view(self.M,in_ch,ksize,ksize)
        self.D=nn.Parameter(D0)
        if untied_w:
            self.W=nn.Parameter(D0.clone())
        self.gamma_raw=nn.Parameter(torch.zeros(n_layers))
        self.theta_raw=nn.Parameter(torch.full((n_layers,self.M),-3.0))
        self.logit_scale_raw=nn.Parameter(torch.tensor(0.5413))
        self.bias=nn.Parameter(torch.zeros(num_classes))

    def _unit_D(self):
        n=self.D.flatten(1).norm(dim=1).view(-1,1,1,1).clamp_min(1e-8)
        return self.D/n

    def _unit_W(self,Dn):
        if not self.untied_w: return Dn
        n=self.W.flatten(1).norm(dim=1).view(-1,1,1,1).clamp_min(1e-8)
        return self.W/n

    def _specific_slice(self,c):
        lo=self.shared_atoms+c*self.apc
        return slice(lo,lo+self.apc)

    def encode(self,fmap):
        Dn=self._unit_D(); Wn=self._unit_W(Dn)
        B,_,H,W=fmap.shape
        Z=fmap.new_zeros(B,self.M,H,W)
        steps=self.step_max*torch.sigmoid(self.gamma_raw)
        thrs=F.softplus(self.theta_raw)
        for k in range(self.n_layers):
            recon=F.conv_transpose2d(Z,Dn,padding=self.pad)
            grad=F.conv2d(fmap-recon,Wn,padding=self.pad)
            Z=Z+steps[k]*grad
            t=thrs[k].view(1,self.M,1,1)
            Z=torch.sign(Z)*F.relu(torch.abs(Z)-t)
        return Z,Dn

    def forward(self,fmap,labels=None):
        Z,Dn=self.encode(fmap)
        if self.shared_atoms>0:
            rs=F.conv_transpose2d(Z[:,:self.shared_atoms],Dn[:self.shared_atoms],padding=self.pad)
        else:
            rs=torch.zeros_like(fmap)
        res=fmap.new_empty(fmap.size(0),self.num_classes)
        for c in range(self.num_classes):
            idx=self._specific_slice(c)
            rc=F.conv_transpose2d(Z[:,idx],Dn[idx],padding=self.pad)
            res[:,c]=(fmap-rs-rc).square().flatten(1).mean(1)
        scale=F.softplus(self.logit_scale_raw)
        return {"logits":-scale*res+self.bias,"residuals":res}


class TeacherCSUNetSSR(nn.Module):
    def __init__(self,num_classes=2,feat_ch=16,target_hw=12,shared_atoms=4,
                 atoms_per_class=6,ksize=5,n_layers=5,step_max=0.15,untied_w=True):
        super().__init__()
        self.num_classes=num_classes
        self.encoder=TeacherNaturalFeatureEncoder(feat_ch,target_hw)
        self.head=TeacherSharedSpecificHead(
            self.encoder.out_ch,num_classes,shared_atoms,atoms_per_class,
            ksize,n_layers,step_max,untied_w
        )

    def forward(self,x,labels=None):
        return self.head(self.encoder(x),labels=labels)


def _strip_module_prefix(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    if sd and all(k.startswith("module.") for k in sd):
        return {k[7:]: v for k,v in sd.items()}
    return sd


def load_teacher_checkpoint(path: str, device: torch.device) -> Tuple[nn.Module, Dict]:
    ck=torch.load(path,map_location="cpu")
    a=ck.get("args",{})
    teacher=TeacherCSUNetSSR(
        num_classes=2,
        feat_ch=int(a.get("feat_ch",16)),
        target_hw=int(a.get("target_hw",12)),
        shared_atoms=int(a.get("shared_atoms",4)),
        atoms_per_class=int(a.get("apc",6)),
        ksize=int(a.get("ksize",5)),
        n_layers=int(a.get("layers",5)),
        step_max=float(a.get("step_max",0.15)),
        untied_w=bool(a.get("untied_w",True)),
    )
    sd=ck.get("model_ema",ck.get("model",ck))
    sd=_strip_module_prefix(sd)
    missing,unexpected=teacher.load_state_dict(sd,strict=False)
    if missing or unexpected:
        raise RuntimeError(
            "Teacher checkpoint architecture mismatch. "
            f"missing={missing[:10]} unexpected={unexpected[:10]}"
        )
    teacher.eval().to(device)
    for p in teacher.parameters(): p.requires_grad_(False)
    print(
        f"[KD] teacher loaded: {path} | bestF1={100*float(ck.get('best_f1',-1)):.2f}% "
        f"epoch={ck.get('epoch','?')}", flush=True
    )
    return teacher, ck


# ============================================================================
# Loss / EMA
# ============================================================================
def ssr_supervised_loss(out: Dict[str,torch.Tensor], labels: torch.Tensor,
                        label_smoothing: float, w_recon: float, w_cross: float,
                        w_margin: float, margin: float, w_sep: float):
    ce=F.cross_entropy(out["logits"],labels,label_smoothing=label_smoothing)
    recon=out.get("recon",ce.new_zeros(()))
    cross=out.get("cross",ce.new_zeros(()))
    sep=out.get("sep",ce.new_zeros(()))
    mr=out.get("margin_raw",ce.new_zeros(labels.shape))
    m_loss=F.relu(mr+margin).mean()
    total=ce+w_recon*recon+w_cross*cross+w_margin*m_loss+w_sep*sep
    return total,{"ce":float(ce.detach()),"recon":float(recon.detach()),
                  "cross":float(cross.detach()),"margin":float(m_loss.detach()),
                  "sep":float(sep.detach())}


def kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, T: float) -> torch.Tensor:
    return (T*T)*F.kl_div(
        F.log_softmax(student_logits/T,dim=1),
        F.softmax(teacher_logits/T,dim=1),
        reduction="batchmean"
    )


class ModelEMA:
    def __init__(self,model:nn.Module,decay:float=0.999):
        self.decay=float(decay)
        self.module=copy.deepcopy(model).eval()
        for p in self.module.parameters(): p.requires_grad_(False)
        self.n=0

    @torch.no_grad()
    def update(self,model:nn.Module):
        self.n+=1
        d=min(self.decay,(1.0+self.n)/(10.0+self.n))
        msd=model.state_dict()
        for k,v in self.module.state_dict().items():
            src=msd[k].detach()
            if v.dtype.is_floating_point: v.mul_(d).add_(src,alpha=1-d)
            else: v.copy_(src)


# ============================================================================
# Metrics / evaluation / train
# ============================================================================
@dataclass
class EvalMetrics:
    accuracy: float
    precision_macro: float
    recall_macro: float
    f1_macro: float
    precision_cat: float
    recall_cat: float
    f1_cat: float
    precision_dog: float
    recall_dog: float
    f1_dog: float
    n: int


def metrics_from_confusion(cm:np.ndarray)->EvalMetrics:
    total=int(cm.sum()); acc=float(np.trace(cm)/max(total,1))
    ps=[];rs=[];fs=[]
    for c in range(cm.shape[0]):
        tp=float(cm[c,c]); fp=float(cm[:,c].sum()-cm[c,c]); fn=float(cm[c,:].sum()-cm[c,c])
        p=tp/max(tp+fp,1e-12); r=tp/max(tp+fn,1e-12); f=2*p*r/max(p+r,1e-12)
        ps.append(p);rs.append(r);fs.append(f)
    return EvalMetrics(acc,float(np.mean(ps)),float(np.mean(rs)),float(np.mean(fs)),
                       ps[0],rs[0],fs[0],ps[1],rs[1],fs[1],total)


@torch.inference_mode()
def evaluate(model:nn.Module,loader:DataLoader,device:torch.device,amp:bool=False):
    model.eval(); cm=np.zeros((2,2),dtype=np.int64)
    use_amp=amp and device.type=="cuda"
    for x,y in loader:
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.autocast(device_type="cuda",enabled=use_amp):
            pred=model(x)["logits"].argmax(1)
        for t,p in zip(y.cpu().numpy(),pred.cpu().numpy()): cm[int(t),int(p)]+=1
    return metrics_from_confusion(cm),cm


def _teacher_logits_chunked(teacher, xt: torch.Tensor, microbatch: int) -> torch.Tensor:
    """No-grad teacher forward in chunks to avoid doubling peak GPU activation memory."""
    if microbatch <= 0 or xt.size(0) <= microbatch:
        return teacher(xt)["logits"]
    outs=[]
    for i in range(0, xt.size(0), microbatch):
        outs.append(teacher(xt[i:i+microbatch])["logits"])
    return torch.cat(outs, dim=0)


def train_one_epoch(model,teacher,loader,opt,device,scaler,args,epoch,ema):
    model.train(); agg={}; n=0; correct=0
    use_amp=scaler is not None
    it=tqdm(loader,desc=f"Epoch {epoch}/{args.epochs}",leave=False,ncols=115) if _HAS_TQDM else loader
    kd_ramp=min(1.0,epoch/max(1,args.kd_ramp_epochs)) if teacher is not None else 0.0
    kd_w=args.kd_weight*kd_ramp
    for xs,xt,y in it:
        xs=xs.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        if teacher is not None: xt=xt.to(device,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda",enabled=use_amp):
            out=model(xs,labels=y)
            sup,parts=ssr_supervised_loss(
                out,y,args.label_smoothing,args.w_recon,args.w_cross,
                args.w_margin,args.margin,args.w_sep
            )
            if teacher is not None and kd_w>0:
                with torch.no_grad(): tlog=_teacher_logits_chunked(teacher,xt,args.teacher_microbatch)
                kd=kd_loss(out["logits"],tlog,args.kd_temp)
                loss=sup+kd_w*kd
            else:
                kd=sup.new_zeros(()); loss=sup
        if use_amp:
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),args.grad_clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),args.grad_clip); opt.step()
        if ema is not None: ema.update(model)
        bs=xs.size(0); n+=bs; correct+=int((out["logits"].argmax(1)==y).sum())
        vals={**parts,"kd":float(kd.detach()),"kd_weight":float(kd_w),"total":float(loss.detach())}
        for k,v in vals.items(): agg[k]=agg.get(k,0.0)+v*bs
        if _HAS_TQDM: it.set_postfix(loss=f"{float(loss.detach()):.3f}",acc=f"{100*correct/max(n,1):.1f}%")
    m={k:v/max(n,1) for k,v in agg.items()}; m["train_acc"]=correct/max(n,1)
    return m


def print_metrics(prefix:str,m:EvalMetrics,cm:np.ndarray):
    print(f"[{prefix}] acc={100*m.accuracy:.2f}% macroP={100*m.precision_macro:.2f}% "
          f"macroR={100*m.recall_macro:.2f}% macroF1={100*m.f1_macro:.2f}% n={m.n}",flush=True)
    print(f"[{prefix}] Cat P/R/F1={100*m.precision_cat:.2f}/{100*m.recall_cat:.2f}/{100*m.f1_cat:.2f}% | "
          f"Dog P/R/F1={100*m.precision_dog:.2f}/{100*m.recall_dog:.2f}/{100*m.f1_dog:.2f}%",flush=True)
    print(f"[{prefix}] CM rows=true cols=pred header=Cat,Dog",flush=True)
    print(f"[{prefix}] Cat,{int(cm[0,0])},{int(cm[0,1])}",flush=True)
    print(f"[{prefix}] Dog,{int(cm[1,0])},{int(cm[1,1])}",flush=True)


def save_confusion_csv(path:str,cm:np.ndarray):
    with open(path,"w",newline="",encoding="utf-8") as f:
        w=csv.writer(f);w.writerow(["true\\pred"]+CLASS_NAMES)
        for i,n in enumerate(CLASS_NAMES): w.writerow([n]+[int(x) for x in cm[i]])


def load_history(path:str)->List[Dict]:
    if not os.path.exists(path): return []
    with open(path,"r",encoding="utf-8") as f: return list(csv.DictReader(f))


def save_history(path:str,history:List[Dict]):
    if not history:return
    keys=sorted(set().union(*[h.keys() for h in history]))
    with open(path,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(history)


# ============================================================================
# Efficiency / deploy graph
# ============================================================================
def count_params(model:nn.Module)->int:
    return sum(p.numel() for p in model.parameters())


def _encoder_macs_and_max_activation(encoder:nn.Module,dummy:torch.Tensor)->Tuple[int,int,Tuple[int,...]]:
    macs=0; max_numel=0; max_shape=(); handles=[]
    def hook(m,inp,out):
        nonlocal macs,max_numel,max_shape
        if not isinstance(out,torch.Tensor): return
        n=out[0].numel() if out.ndim>=1 else out.numel()
        if n>max_numel: max_numel=n; max_shape=tuple(out.shape[1:])
        if isinstance(m,nn.Conv2d):
            B,Cout,H,W=out.shape;kh,kw=m.kernel_size;cinpg=m.in_channels//m.groups
            macs+=int(B*Cout*H*W*cinpg*kh*kw)
    for mod in encoder.modules():
        if isinstance(mod,nn.Conv2d): handles.append(mod.register_forward_hook(hook))
    with torch.inference_mode(): encoder(dummy)
    for h in handles:h.remove()
    return macs,max_numel,max_shape


def estimate_model_cost(model:CSUNetSSRS3,img_size:int,device:torch.device):
    model.eval();x=torch.zeros(1,3,img_size,img_size,device=device)
    enc_macs,max_act,max_shape=_encoder_macs_and_max_activation(model.encoder,x)
    with torch.inference_mode(): fmap=model.encoder(x)
    _,C,H,W=fmap.shape;head=model.head;k2=head.ksize**2
    base=H*W*head.M*C*k2
    head_macs=head.n_layers*2*base+base
    total=int(enc_macs+head_macs)
    return total,(C,H,W),max_act,max_shape,enc_macs,int(head_macs)


@torch.inference_mode()
def benchmark_inference(model,img_size,device,amp=False,warmup=20,reps=100):
    model.eval();x=torch.randn(1,3,img_size,img_size,device=device);use_amp=amp and device.type=="cuda"
    if device.type=="cuda": torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)
    for _ in range(warmup):
        with torch.autocast(device_type="cuda",enabled=use_amp): model(x)
    if device.type=="cuda":torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(reps):
        with torch.autocast(device_type="cuda",enabled=use_amp):model(x)
    if device.type=="cuda":torch.cuda.synchronize()
    ms=1000*(time.perf_counter()-t0)/reps
    r={"latency_ms_b1":ms,"fps_b1":1000/max(ms,1e-9)}
    if device.type=="cuda":r["peak_vram_mb_b1"]=torch.cuda.max_memory_allocated(device)/(1024**2)
    try:
        import psutil;r["process_rss_mb"]=psutil.Process(os.getpid()).memory_info().rss/(1024**2)
    except Exception:pass
    return r


class FrozenS3Head(nn.Module):
    """Frozen deploy head: no normalize/sigmoid/softplus; synthesis uses Conv2d only."""
    def __init__(self,head:SharedSpecificS3Head):
        super().__init__();head.eval()
        with torch.no_grad():
            Dn=head._unit_D().detach().clone()
            Wn=head._unit_W(Dn).detach().clone()
            # conv_transpose2d(z, Dn, stride=1) == conv2d(z, flipped-permuted Dn)
            Dsyn=Dn.permute(1,0,2,3).flip(-1,-2).contiguous()
            steps=(head.step_max*torch.sigmoid(head.gamma_raw)).detach().clone()
            thrs=F.softplus(head.theta_raw).detach().clone()
            scale=F.softplus(head.logit_scale_raw).detach().clone()
        self.register_buffer("Dsyn",Dsyn)
        self.register_buffer("Wn",Wn)
        self.register_buffer("steps",steps)
        self.register_buffer("thrs",thrs)
        self.register_buffer("scale",scale)
        self.register_buffer("bias",head.bias.detach().clone())
        self.shared_atoms=head.shared_atoms;self.apc=head.apc;self.num_classes=head.num_classes
        self.M=head.M;self.n_layers=head.n_layers;self.pad=head.pad

    def _specific_slice(self,c:int)->slice:
        lo=self.shared_atoms+c*self.apc;return slice(lo,lo+self.apc)

    def _synth(self,Z:torch.Tensor,atom_slice:slice)->torch.Tensor:
        return F.conv2d(Z[:,atom_slice],self.Dsyn[:,atom_slice],padding=self.pad)

    def forward(self,fmap:torch.Tensor)->torch.Tensor:
        B,_,H,W=fmap.shape;Z=fmap.new_zeros(B,self.M,H,W)
        for k in range(self.n_layers):
            recon=F.conv2d(Z,self.Dsyn,padding=self.pad)
            grad=F.conv2d(fmap-recon,self.Wn,padding=self.pad)
            Z=Z+self.steps[k]*grad
            t=self.thrs[k].view(1,self.M,1,1)
            Z=soft_threshold_esp(Z,t)
        if self.shared_atoms>0:
            rs=self._synth(Z,slice(0,self.shared_atoms))
        else:
            rs=torch.zeros_like(fmap)
        vals=[]
        for c in range(self.num_classes):
            rc=self._synth(Z,self._specific_slice(c))
            vals.append((fmap-rs-rc).square().flatten(1).mean(1))
        res=torch.stack(vals,dim=1)
        return -self.scale*res+self.bias


class S3DeployModel(nn.Module):
    def __init__(self,student:CSUNetSSRS3):
        super().__init__()
        self.encoder=copy.deepcopy(student.encoder).eval()
        self.head=FrozenS3Head(student.head)

    def forward(self,x:torch.Tensor)->torch.Tensor:
        return self.head(self.encoder(x))


@torch.inference_mode()
def verify_deploy_equivalence(student:CSUNetSSRS3,deploy:S3DeployModel,img_size:int,device):
    student.eval();deploy.eval().to(device)
    x=torch.randn(2,3,img_size,img_size,device=device)
    a=student(x)["logits"];b=deploy(x)
    diff=float((a-b).abs().max().cpu())
    print(f"[DEPLOY] frozen graph max_abs_logit_diff={diff:.3e}",flush=True)
    if diff>1e-3: raise RuntimeError(f"Deploy graph equivalence failed: {diff}")
    return diff


def export_onnx_deploy(student:CSUNetSSRS3,path:str,img_size:int,device)->Optional[float]:
    deploy=S3DeployModel(student).eval().to(device)
    verify_deploy_equivalence(student,deploy,img_size,device)
    x=torch.randn(1,3,img_size,img_size,device=device)
    try:
        torch.onnx.export(
            deploy,x,path,input_names=["image"],output_names=["logits"],
            opset_version=18,do_constant_folding=True,dynamo=False
        )
        mb=os.path.getsize(path)/(1024**2)
        print(f"[EXPORT] static ONNX opset18 = {mb:.3f} MB -> {path}",flush=True)
        return mb
    except Exception as e:
        print(f"[EXPORT] ONNX export failed: {e}",flush=True);return None


def save_deploy_state(student:CSUNetSSRS3,path:str)->float:
    deploy=S3DeployModel(student).cpu().eval()
    torch.save(deploy.state_dict(),path)
    return os.path.getsize(path)/(1024**2)


# ============================================================================
# Checkpoint
# ============================================================================
def save_checkpoint(path,model,ema,opt,sched,epoch,best_f1,no_improve,args,metrics):
    obj={"model":model.state_dict(),"epoch":epoch,"best_f1":best_f1,
         "no_improve":no_improve,"args":vars(args),"metrics":asdict(metrics),
         "optimizer":opt.state_dict(),"scheduler":sched.state_dict(),"classes":CLASS_NAMES}
    if ema is not None:obj["model_ema"]=ema.module.state_dict()
    torch.save(obj,path)


def load_checkpoint(path,model,ema=None,opt=None,sched=None,device="cpu",resume=False):
    ck=torch.load(path,map_location=device);model.load_state_dict(_strip_module_prefix(ck["model"]),strict=True)
    if ema is not None and "model_ema" in ck:ema.module.load_state_dict(_strip_module_prefix(ck["model_ema"]),strict=True)
    if resume and opt is not None and "optimizer" in ck:opt.load_state_dict(ck["optimizer"])
    if resume and sched is not None and "scheduler" in ck:sched.load_state_dict(ck["scheduler"])
    return ck


def get_eval_model(model,ema):return ema.module if ema is not None else model


def parse_widths(s:str)->Tuple[int,int,int,int,int]:
    vals=tuple(int(x.strip()) for x in s.split(",") if x.strip())
    if len(vals)!=5 or min(vals)<=0:raise ValueError("--widths must contain five positive ints, e.g. 16,24,32,48,64")
    return vals


# ============================================================================
# CLI / main
# ============================================================================
def build_argparser():
    p=argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    # data
    p.add_argument("--data_root",type=str,default="/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages")
    p.add_argument("--out_dir",type=str,default="/kaggle/working/csunet_ssr_s3")
    p.add_argument("--img_size",type=int,default=128)
    p.add_argument("--teacher_img_size",type=int,default=160)
    p.add_argument("--val_frac",type=float,default=0.10);p.add_argument("--test_frac",type=float,default=0.10)
    p.add_argument("--workers",type=int,default=4);p.add_argument("--subset_frac",type=float,default=1.0)
    p.add_argument("--erase_p",type=float,default=0.0,help="Keep 0 initially; enable only if train-val gap persists")
    # model
    p.add_argument("--widths",type=str,default="16,24,32,48,64")
    p.add_argument("--feat_ch",type=int,default=12);p.add_argument("--shared_atoms",type=int,default=2)
    p.add_argument("--apc",type=int,default=4);p.add_argument("--ksize",type=int,default=3);p.add_argument("--layers",type=int,default=4)
    p.add_argument("--step_max",type=float,default=0.15);p.add_argument("--untied_w",action=argparse.BooleanOptionalAction,default=True)
    # optimization / overfit
    p.add_argument("--epochs",type=int,default=70);p.add_argument("--batch_size",type=int,default=128)
    p.add_argument("--lr",type=float,default=1.5e-3);p.add_argument("--weight_decay",type=float,default=5e-4)
    p.add_argument("--warmup_epochs",type=int,default=4);p.add_argument("--label_smoothing",type=float,default=0.05)
    p.add_argument("--grad_clip",type=float,default=1.0);p.add_argument("--ema_decay",type=float,default=0.999)
    p.add_argument("--patience",type=int,default=8);p.add_argument("--min_delta",type=float,default=0.001,
                   help="Validation macro-F1 must improve by at least this amount to reset patience")
    p.add_argument("--gap_warn",type=float,default=0.03,help="Warn if trainAcc-valAcc exceeds this")
    p.add_argument("--amp",action=argparse.BooleanOptionalAction,default=True)
    # SSR loss
    p.add_argument("--w_recon",type=float,default=0.05);p.add_argument("--w_cross",type=float,default=0.05)
    p.add_argument("--w_margin",type=float,default=0.20);p.add_argument("--margin",type=float,default=0.10);p.add_argument("--w_sep",type=float,default=0.01)
    # KD
    p.add_argument("--teacher_ckpt",type=str,default="")
    p.add_argument("--kd_weight",type=float,default=0.40);p.add_argument("--kd_temp",type=float,default=3.0)
    p.add_argument("--kd_ramp_epochs",type=int,default=5)
    p.add_argument("--teacher_microbatch",type=int,default=64,help="Chunk teacher forward to reduce KD peak VRAM; 0 disables chunking")
    # control
    p.add_argument("--seed",type=int,default=42);p.add_argument("--resume",action="store_true")
    p.add_argument("--eval_only",action="store_true");p.add_argument("--unlock_test",action="store_true")
    p.add_argument("--export_onnx",action="store_true");p.add_argument("--benchmark_cpu",action="store_true")
    p.add_argument("--smoke_test",action="store_true");p.add_argument("--nondeterministic",action="store_true")
    return p


def main(argv=None):
    args=build_argparser().parse_args(argv);widths=parse_widths(args.widths)
    if args.img_size%16!=0:
        raise ValueError("For the S3 encoder use img_size divisible by 16 (recommended 128)")
    os.makedirs(args.out_dir,exist_ok=True);set_seed(args.seed,deterministic=not args.nondeterministic)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device={device} seed={args.seed} deterministic={not args.nondeterministic}",flush=True)

    samples=discover_catdog_samples(args.data_root,args.out_dir);samples=maybe_subset(samples,args.subset_frac,args.seed)
    cnt=[sum(int(y==c) for _,y in samples) for c in range(2)]
    print(f"[DATA] labeled total={len(samples)} Cat={cnt[0]} Dog={cnt[1]}",flush=True)
    tr_s,va_s,te_s=stratified_split(samples,args.seed,args.val_frac,args.test_frac)
    print_split_stats("train",tr_s);print_split_stats("val",va_s);print_split_stats("test",te_s)

    train_ds=TrainPairDataset(tr_s,PairedTrainTransform(args.img_size,args.teacher_img_size,args.erase_p))
    eval_tf=StudentEvalTransform(args.img_size);val_ds=EvalDataset(va_s,eval_tf);test_ds=EvalDataset(te_s,eval_tf)
    g=make_generator(args.seed)
    common=dict(num_workers=args.workers,pin_memory=(device.type=="cuda"),worker_init_fn=seed_worker,
                persistent_workers=(args.workers>0))
    if args.workers>0:common["prefetch_factor"]=2
    tl=DataLoader(train_ds,batch_size=args.batch_size,shuffle=True,drop_last=True,generator=g,**common)
    vl=DataLoader(val_ds,batch_size=args.batch_size,shuffle=False,drop_last=False,**common)
    tsl=DataLoader(test_ds,batch_size=args.batch_size,shuffle=False,drop_last=False,**common)

    model=CSUNetSSRS3(2,args.feat_ch,widths,args.shared_atoms,args.apc,args.ksize,args.layers,args.step_max,args.untied_w).to(device)
    n_params=count_params(model)
    macs,fmap_shape,max_act,max_act_shape,enc_macs,head_macs=estimate_model_cost(model,args.img_size,device)
    print(f"[REPORT] PARAMS={n_params:,}",flush=True)
    print(f"[REPORT] FEATURE_MAP=CxHxW={fmap_shape[0]}x{fmap_shape[1]}x{fmap_shape[2]}",flush=True)
    print(f"[REPORT] MACS={macs/1e6:.3f}M (encoder={enc_macs/1e6:.3f}M head={head_macs/1e6:.3f}M) FLOPS~={2*macs/1e6:.3f}M",flush=True)
    print(f"[REPORT] MAX_SINGLE_ENCODER_ACT={max_act:,} elements shape={max_act_shape} ~{max_act/1024:.1f} KiB @INT8",flush=True)
    print(f"[REPORT] RAW_INT8_WEIGHT_LOWER_BOUND~={n_params/1024:.1f} KiB (actual .espdl will be larger)",flush=True)

    teacher=None;teacher_ck={}
    if args.teacher_ckpt and not args.eval_only:
        teacher,teacher_ck=load_teacher_checkpoint(args.teacher_ckpt,device)
        ta=teacher_ck.get("args",{})
        for key,cur in (("seed",args.seed),("val_frac",args.val_frac),("test_frac",args.test_frac)):
            if key in ta and str(ta[key])!=str(cur):
                print(f"[KD][WARN] teacher {key}={ta[key]} vs student {key}={cur}; splits/views may not match",flush=True)
    elif args.kd_weight>0 and not args.eval_only:
        print("[KD] no --teacher_ckpt: KD disabled for this run",flush=True)

    if args.smoke_test:
        model.train();xs=torch.randn(4,3,args.img_size,args.img_size,device=device);y=torch.tensor([0,1,0,1],device=device)
        out=model(xs,labels=y);sup,parts=ssr_supervised_loss(out,y,args.label_smoothing,args.w_recon,args.w_cross,args.w_margin,args.margin,args.w_sep)
        loss=sup
        if teacher is not None:
            xt=torch.randn(4,3,args.teacher_img_size,args.teacher_img_size,device=device)
            with torch.no_grad():tlog=_teacher_logits_chunked(teacher,xt,args.teacher_microbatch)
            kd=kd_loss(out["logits"],tlog,args.kd_temp);loss=loss+args.kd_weight*kd;parts["kd"]=float(kd.detach())
        loss.backward();finite=all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None)
        print(f"[SMOKE] logits={tuple(out['logits'].shape)} loss={float(loss.detach()):.6f} parts={parts} gradients_finite={finite}",flush=True)
        deploy=S3DeployModel(model).to(device);verify_deploy_equivalence(model,deploy,args.img_size,device)
        if not finite:raise RuntimeError("non-finite gradients")
        return

    opt=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=args.weight_decay)
    if args.warmup_epochs>0:
        warm=torch.optim.lr_scheduler.LinearLR(opt,start_factor=0.1,end_factor=1.0,total_iters=args.warmup_epochs)
        cos=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,args.epochs-args.warmup_epochs),eta_min=args.lr*0.01)
        sched=torch.optim.lr_scheduler.SequentialLR(opt,[warm,cos],milestones=[args.warmup_epochs])
    else:
        sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,args.epochs),eta_min=args.lr*0.01)
    ema=ModelEMA(model,args.ema_decay) if args.ema_decay>0 else None
    scaler=torch.amp.GradScaler("cuda") if (args.amp and device.type=="cuda") else None

    best_path=os.path.join(args.out_dir,"csunet_ssr_s3_best.pt");last_path=os.path.join(args.out_dir,"csunet_ssr_s3_last.pt")
    hist_path=os.path.join(args.out_dir,"history.csv")
    start_ep=1;best_f1=-1.0;no_improve=0;history=[]
    if args.resume:
        lp=last_path if os.path.exists(last_path) else best_path
        if not os.path.exists(lp):raise FileNotFoundError(f"--resume but checkpoint missing in {args.out_dir}")
        ck=load_checkpoint(lp,model,ema,opt,sched,device,True);start_ep=int(ck.get("epoch",0))+1
        best_f1=float(ck.get("best_f1",-1));no_improve=int(ck.get("no_improve",0));history=load_history(hist_path)
        print(f"[RESUME] {lp} epoch={start_ep-1} bestF1={100*best_f1:.2f}% noImprove={no_improve}",flush=True)

    if args.eval_only:
        if not os.path.exists(best_path):raise FileNotFoundError(best_path)
        ck=torch.load(best_path,map_location=device);model.load_state_dict(_strip_module_prefix(ck.get("model_ema",ck["model"])),strict=True)
        vm,vcm=evaluate(model,vl,device,args.amp);print_metrics("VAL",vm,vcm)
        if args.unlock_test:
            tm,tcm=evaluate(model,tsl,device,args.amp);print_metrics("TEST",tm,tcm);save_confusion_csv(os.path.join(args.out_dir,"confusion_test.csv"),tcm)
        if args.export_onnx:
            export_onnx_deploy(model,os.path.join(args.out_dir,"csunet_ssr_s3_deploy.onnx"),args.img_size,device)
        return

    t_all=time.time()
    for ep in range(start_ep,args.epochs+1):
        t0=time.time();tr=train_one_epoch(model,teacher,tl,opt,device,scaler,args,ep,ema);sched.step()
        em=get_eval_model(model,ema);vm,vcm=evaluate(em,vl,device,args.amp);dt=time.time()-t0
        gap=tr["train_acc"]-vm.accuracy
        row={"epoch":ep,"lr":opt.param_groups[0]["lr"],"seconds":dt,"train_val_gap":gap,
             **{f"train_{k}":v for k,v in tr.items()},**{f"val_{k}":v for k,v in asdict(vm).items()}}
        history.append(row);save_history(hist_path,history)
        print(f"[EP {ep:03d}/{args.epochs}] loss={tr['total']:.4f} ce={tr['ce']:.4f} kd={tr['kd']:.4f} "
              f"kdW={tr['kd_weight']:.3f} recon={tr['recon']:.4f} cross={tr['cross']:.4f} margin={tr['margin']:.4f} "
              f"sep={tr['sep']:.5f} trainAcc={100*tr['train_acc']:.2f}% | valAcc={100*vm.accuracy:.2f}% "
              f"valF1={100*vm.f1_macro:.2f}% gap={100*gap:+.2f}% {dt:.0f}s",flush=True)
        if gap>args.gap_warn:
            print(f"[OVERFIT-WARN] train-val gap {100*gap:.2f}% > {100*args.gap_warn:.1f}%. Do not enlarge model; consider erase_p=0.05 only if persistent.",flush=True)
        improved=vm.f1_macro>best_f1+args.min_delta
        if improved:
            best_f1=vm.f1_macro;no_improve=0
            save_checkpoint(best_path,model,ema,opt,sched,ep,best_f1,no_improve,args,vm)
            save_confusion_csv(os.path.join(args.out_dir,"confusion_val_best.csv"),vcm)
            with open(os.path.join(args.out_dir,"metrics_val_best.json"),"w",encoding="utf-8") as f:json.dump(asdict(vm),f,indent=2)
            print(f"        -> saved BEST macro-F1={100*best_f1:.2f}% (min_delta={100*args.min_delta:.2f} pp)",flush=True)
        else:no_improve+=1
        save_checkpoint(last_path,model,ema,opt,sched,ep,best_f1,no_improve,args,vm)
        if no_improve>=args.patience:
            print(f"[EARLY STOP] no >= {100*args.min_delta:.2f} pp F1 improvement for {args.patience} epochs",flush=True);break

    print(f"[TRAIN] finished {(time.time()-t_all)/60:.1f} min | best valF1={100*best_f1:.2f}%",flush=True)
    ck=torch.load(best_path,map_location=device);model.load_state_dict(_strip_module_prefix(ck.get("model_ema",ck["model"])),strict=True);model.eval()
    vm,vcm=evaluate(model,vl,device,args.amp);print_metrics("VAL-BEST",vm,vcm);save_confusion_csv(os.path.join(args.out_dir,"confusion_val_final.csv"),vcm)

    report={"classes":CLASS_NAMES,"params":n_params,"macs":macs,"encoder_macs":enc_macs,"head_macs":head_macs,
            "flops_estimate":2*macs,"feature_map":list(fmap_shape),"max_single_encoder_activation_elements":max_act,
            "max_single_encoder_activation_shape":list(max_act_shape),"int8_single_activation_kib":max_act/1024,
            "raw_int8_weight_lower_bound_kib":n_params/1024,"val":asdict(vm),"test_locked":not args.unlock_test,"config":vars(args)}
    if args.unlock_test:
        tm,tcm=evaluate(model,tsl,device,args.amp);print_metrics("TEST",tm,tcm);save_confusion_csv(os.path.join(args.out_dir,"confusion_test.csv"),tcm)
        with open(os.path.join(args.out_dir,"metrics_test.json"),"w",encoding="utf-8") as f:json.dump(asdict(tm),f,indent=2)
        report["test"]=asdict(tm)
    else:print("[TEST] LOCKED. Unlock only after model/hyperparameters are frozen.",flush=True)

    deploy_state=os.path.join(args.out_dir,"csunet_ssr_s3_frozen_deploy_state.pt")
    deploy_mb=save_deploy_state(model,deploy_state);report["frozen_deploy_state_mb"]=deploy_mb
    print(f"[REPORT] frozen deploy state={deploy_mb:.3f} MB",flush=True)
    verify_deploy_equivalence(model,S3DeployModel(model).to(device),args.img_size,device)
    bench=benchmark_inference(model,args.img_size,device,amp=False);report["runtime_fp32"]=bench
    print("[REPORT] runtime FP32 "+" ".join(f"{k}={v:.3f}" for k,v in bench.items()),flush=True)
    if args.benchmark_cpu and device.type!="cpu":
        cm=copy.deepcopy(model).cpu().eval();cb=benchmark_inference(cm,args.img_size,torch.device("cpu"),False,5,30);report["runtime_cpu_fp32"]=cb
        print("[REPORT] CPU FP32 "+" ".join(f"{k}={v:.3f}" for k,v in cb.items()),flush=True)
    if args.export_onnx:
        mb=export_onnx_deploy(model,os.path.join(args.out_dir,"csunet_ssr_s3_deploy.onnx"),args.img_size,device)
        if mb is not None:report["onnx_fp32_mb"]=mb
    with open(os.path.join(args.out_dir,"final_report.json"),"w",encoding="utf-8") as f:json.dump(report,f,indent=2)
    print(f"[DONE] outputs -> {args.out_dir}",flush=True)


if __name__=="__main__":
    main()


Overwriting csunet_catdog_s3.py


In [8]:
!python /kaggle/working/csunet_catdog_s3.py \
  --data_root /kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages \
  --teacher_ckpt /kaggle/working/csunet_ssr/csunet_ssr_best.pt \
  --out_dir /kaggle/working/csunet_ssr_s3 \
  --img_size 128 \
  --teacher_img_size 160 \
  --widths 16,24,32,48,64 \
  --feat_ch 12 \
  --shared_atoms 2 \
  --apc 4 \
  --ksize 3 \
  --layers 4 \
  --step_max 0.15 \
  --untied_w \
  --batch_size 128 \
  --epochs 70 \
  --lr 1.5e-3 \
  --weight_decay 5e-4 \
  --warmup_epochs 4 \
  --label_smoothing 0.05 \
  --kd_weight 0.40 \
  --kd_temp 3.0 \
  --kd_ramp_epochs 5 \
  --teacher_microbatch 64 \
  --ema_decay 0.999 \
  --patience 8 \
  --min_delta 0.001 \
  --w_recon 0.05 \
  --w_cross 0.05 \
  --w_margin 0.20 \
  --margin 0.10 \
  --w_sep 0.01 \
  --amp

[INFO] device=cuda seed=42 deterministic=True
[DATA] labeled total=24998 Cat=12499 Dog=12499
[DATA] train:  19998 | Cat=  9999 Dog=  9999
[DATA] val  :   2500 | Cat=  1250 Dog=  1250
[DATA] test :   2500 | Cat=  1250 Dog=  1250
[REPORT] PARAMS=20,607
[REPORT] FEATURE_MAP=CxHxW=12x8x8
[REPORT] MACS=10.284M (encoder=9.661M head=0.622M) FLOPS~=20.567M
[REPORT] MAX_SINGLE_ENCODER_ACT=98,304 elements shape=(24, 64, 64) ~96.0 KiB @INT8
[REPORT] RAW_INT8_WEIGHT_LOWER_BOUND~=20.1 KiB (actual .espdl will be larger)
[KD] teacher loaded: /kaggle/working/csunet_ssr/csunet_ssr_best.pt | bestF1=95.72% epoch=38
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read               
  warnings.warn(str(msg))
[EP 001/70] loss=0.8528 ce=0.6914 kd=1.2727 kdW=0.080 recon=0.3719 cross=0.4518 margin=0.0920 sep=0.00011 trainAcc=57.73% | valAcc=61.24% valF1=60.34% gap=-3.51% 72s
        -> saved BEST macro-F1=60.34% (min_delta=0.10 pp)
[EP 002/70] loss=0.9366 ce=0.6